# Part 5B — GraphRAG (Notebook 06)

This notebook adds graph-structured retrieval expansion over the same 4,000-paper corpus.


## Tutorial Goals

This notebook is a standalone, zero-to-hero tutorial with:

1. Concept explanation from first principles
2. Architecture and workflow breakdown
3. End-to-end implementation code
4. Real execution outputs and benchmark metrics
5. Practical analysis and production takeaways


## What is this technique?

        ### Definition and core concepts
        GraphRAG connects documents through entity/theme relationships and uses these links during retrieval.

        ### Why was this technique developed?
        Top-k vector retrieval can miss connected evidence that is not nearest by embedding similarity.

        ### What limitations of traditional RAG does it solve?
        It improves multi-hop evidence coverage and broad synthesis support.

        ### Architecture and workflow diagram explanation

```mermaid
graph TD
    Q[Query] --> S[Seed Retrieval]
    S --> E[Entity Expansion]
    E --> M[Merged Context]
    M --> G[Generation]
```


        ### Component-by-component breakdown
        Seed retriever, entity graph, expansion policy, merged ranking, generation/evaluation.

        ### When should it be used in real-world systems?
        Use when relationships across documents are important for answer completeness.

        ### Advantages and disadvantages
        **Advantages**
        - Better context completeness
- Improves relationship-aware retrieval
- Better for broad/synthesis queries

        **Disadvantages**
        - Graph construction cost
- Added pipeline complexity
- More tuning surfaces

        ### Comparison against standard RAG and other implemented RAG variants
        Compared with Hybrid RAG, GraphRAG is more relation-aware. Compared with Agentic RAG, it is less adaptive but structurally richer.

        ### Implementation details and design decisions used in this project
        This project uses deterministic lightweight entity extraction to keep the 4,000-paper run practical.


In [ ]:
from __future__ import annotations

import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = Path('.').resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.rag_v2.data import load_base_corpus, load_papers_from_chunks
from src.rag_v2.retrieval import DenseRetriever, BM25Retriever, HybridRetriever
from src.rag_v2.metrics import build_keyword_eval_set, compute_retrieval_metrics, save_json


from src.rag_v2.graph import build_paper_entity_graph, build_entity_to_papers, expand_with_graph
ART = PROJECT_ROOT / 'artifacts' / 'rag_v2'
ART.mkdir(parents=True, exist_ok=True)

index, chunks = load_base_corpus()
papers = load_papers_from_chunks(chunks)

display(Markdown(f"Loaded **{len(chunks):,} chunks** from **{len(papers):,} papers** (FAISS dim={index.d})."))


In [ ]:
dense = DenseRetriever(index=index, chunks=chunks)
bm25 = BM25Retriever(chunks=chunks)
hybrid = HybridRetriever(dense=dense, bm25=bm25, alpha=0.7)

graph, paper_to_entities = build_paper_entity_graph(papers, max_entities_per_paper=8)
entity_to_papers = build_entity_to_papers(paper_to_entities)

paper_to_chunks = {}
for c in chunks:
    paper_to_chunks.setdefault(c['paper_id'], []).append(c)

class GraphRetriever:
    def __init__(self, base):
        self.base = base

    def retrieve(self, query: str, k: int = 10):
        seed = self.base.retrieve(query, k=6)
        seed_papers = list(dict.fromkeys(r['paper_id'] for r in seed))
        extra = expand_with_graph(seed_papers, paper_to_entities, entity_to_papers, max_extra_papers=10)
        merged = list(seed)
        for pid in extra:
            for c in paper_to_chunks.get(pid, [])[:1]:
                merged.append({**c, 'score': 0.05, 'retriever': 'graph_expand'})
        merged.sort(key=lambda x: x.get('score', 0.0), reverse=True)
        seen, out = set(), []
        for row in merged:
            if row['chunk_id'] in seen:
                continue
            out.append(row)
            seen.add(row['chunk_id'])
            if len(out) >= k:
                break
        return out

graph_retriever = GraphRetriever(hybrid)

eval_set = build_keyword_eval_set(papers)
graph_results = compute_retrieval_metrics(eval_set, {'Hybrid': hybrid, 'GraphRAG': graph_retriever}, k=5)
graph_df = pd.DataFrame(graph_results).T
graph_df


In [ ]:
out_json = ART / 'graphrag' / '06_graphrag_metrics.json'
out_json.parent.mkdir(parents=True, exist_ok=True)
entity_nodes = sum(1 for _, d in graph.nodes(data=True) if d.get('node_type') == 'entity')
save_json(out_json, {'retrieval': graph_results, 'graph': {'paper_nodes': len(papers), 'entity_nodes': entity_nodes, 'edges': graph.number_of_edges()}})
out_json


In [ ]:
delta_recall = graph_df.loc['GraphRAG', 'recall@5'] - graph_df.loc['Hybrid', 'recall@5']
delta_mrr = graph_df.loc['GraphRAG', 'mrr'] - graph_df.loc['Hybrid', 'mrr']

if delta_recall == 0 and delta_mrr == 0:
    graph_note = 'Graph expansion preserved baseline quality but did not improve weak-supervision retrieval metrics in this run.'
else:
    graph_note = 'Graph expansion changed retrieval quality versus Hybrid; inspect query-level errors for where gains/losses occur.'

analysis = (
    "## Post-run Analysis (Real Results)\n\n"
    f"- Paper nodes: **{len(papers):,}**\n"
    f"- Entity nodes: **{entity_nodes:,}**\n"
    f"- Graph edges: **{graph.number_of_edges():,}**\n"
    f"- Recall@5 delta vs Hybrid: **{delta_recall:+.4f}**\n"
    f"- MRR delta vs Hybrid: **{delta_mrr:+.4f}**\n"
    f"- GraphRAG P50 latency: **{graph_df.loc['GraphRAG','latency_p50_ms']:.2f} ms**\n"
    f"- GraphRAG P95 latency: **{graph_df.loc['GraphRAG','latency_p95_ms']:.2f} ms**\n\n"
    f"- Observation: {graph_note}"
)
display(Markdown(analysis))
